Dado que el entrenamiento de redes neuronales es una tarea  muy costosa, **se recomienda ejecutar el notebooks en [Google Colab](https://colab.research.google.com)**, por supuesto también se puede ejecutar en local.

Al entrar en [Google Colab](https://colab.research.google.com) bastará con hacer click en `upload` y subir este notebook. No olvide luego descargarlo en `File->Download .ipynb`

**El examen deberá ser entregado con las celdas ejecutadas, si alguna celda no está ejecutadas no se contará.**

El examen se divide en tres partes, con la puntuación que se indica a continuación. La puntuación máxima será 10.

    
- [Actividad 1: Redes Recurrentes](#actividad_1): 10 pts
    - [Cuestión 1](#3.1): 2.5 pt
    - [Cuestión 2](#3.2): 2.5 pt
    - [Cuestión 3](#3.3): 2.5 pts
    - [Cuestión 4](#3.4): 1.25 pts
    - [Cuestión 5](#3.5): 1.25 pts



In [60]:
import tensorflow as tf
from tensorflow import keras
from keras import layers
import pandas as pd
import numpy as np

<a name='actividad_1'></a>
# Actividad 1: Redes Recurrentes


- [Cuestión 1](#3.1): 2.5 pt
- [Cuestión 2](#3.2): 2.5 pt
- [Cuestión 3](#3.3): 2.5 pts
- [Cuestión 4](#3.4): 1.25 pts
- [Cuestión 5](#3.5): 1.25 pts

Vamos a usar un dataset de las temperaturas mínimas diarias en Melbourne. La tarea será la de predecir la temperatura mínima en dos días. Puedes usar técnicas de series temporales vistas en otras asignaturas, pero no es necesario.


In [61]:
dataset_url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv'
data_dir = tf.keras.utils.get_file('daily-min-temperatures.csv', origin=dataset_url)

In [62]:
df = pd.read_csv(data_dir, parse_dates=['Date'])
df.head()

,Date,Temp
0,1981-01-01,20.7
1,1981-01-02,17.9
2,1981-01-03,18.8
3,1981-01-04,14.6
4,1981-01-05,15.8


In [63]:
temperatures = df['Temp'].values
print('number of samples:', len(temperatures))
train_data = temperatures[:3000]
test_data = temperatures[3000:]
print('number of train samples:', len(train_data))
print('number of test samples:', len(test_data))
print('firsts trainn samples:', train_data[:10])

number of samples: 3650
number of train samples: 3000
number of test samples: 650
firsts trainn samples: [20.7 17.9 18.8 14.6 15.8 15.8 15.8 17.4 21.8 20. ]


<a name='3.1'></a>
## Cuestión 1: Convierta `train_data` y `test_data`  en ventanas de tamaño 5, para predecir el valor en 2 días

En la nomenclatura de [Introduction_to_RNN_Time_Series.ipynb](https://github.com/ezponda/intro_deep_learning/blob/main/class/RNN/Introduction_to_RNN_Time_Series.ipynb)
```python
past, future = (5, 2)
```

Para las primeras 10 muestras de train_data `[20.7, 17.9, 18.8, 14.6, 15.8, 15.8, 15.8, 17.4, 21.8, 20. ]` el resultado debería ser:

```python
x[0] : [20.7, 17.9, 18.8, 14.6, 15.8] , y[0]: 15.8
x[1] : [17.9, 18.8, 14.6, 15.8, 15.8] , y[1]: 17.4
x[2] : [18.8, 14.6, 15.8, 15.8, 15.8] , y[2]: 21.8
x[3] : [14.6, 15.8, 15.8, 15.8, 17.4] , y[3]: 20.             
```

In [64]:
def create_windows_np(data, window_size, horizon, shuffle=False):
    """
    Creates a dataset from the given time series data using NumPy.
    
    Parameters:
    data (np.ndarray): Time series data with one dimension.
    window_size (int): The number of past time steps to use as input features.
    horizon (int): The number of future time steps to predict.
    shuffle (bool): Shuffle the windows or not.
    
    Returns:
    tuple: A tuple containing the input-output pairs (windows, targets) as NumPy arrays.
    """

    X, y = [], []
    for i in range(len(data) - window_size - horizon + 1):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size+horizon-1])

    X, y = np.array(X), np.array(y)
    
    if shuffle:
        indices = np.arange(len(X))
        np.random.shuffle(indices)
        X, y = X[indices], y[indices]
    
    return X, y

In [65]:
past, future = (5, 2)
X_train, y_train = create_windows_np(train_data,
                                     window_size=past,
                                     horizon=future,
                                     shuffle=True)
X_test, y_test = create_windows_np(test_data,
                                   window_size=past,
                                   horizon=future,
                                   shuffle=False)

In [66]:
print("x[0] : {}, y[0]: {}".format(train_data[:5].tolist(), train_data[5]))
print("x[1] : {}, y[1]: {}".format(train_data[1:6].tolist(), train_data[6]))
print("x[2] : {}, y[2]: {}".format(train_data[2:7].tolist(), train_data[7]))
print("x[3] : {}, y[3]: {}".format(train_data[3:8].tolist(), train_data[8]))

x[0] : [20.7, 17.9, 18.8, 14.6, 15.8], y[0]: 15.8
x[1] : [17.9, 18.8, 14.6, 15.8, 15.8], y[1]: 15.8
x[2] : [18.8, 14.6, 15.8, 15.8, 15.8], y[2]: 17.4
x[3] : [14.6, 15.8, 15.8, 15.8, 17.4], y[3]: 21.8


<a name='3.2'></a>
## Cuestión 2: Cree un modelo recurrente de dos capas GRU para predecir con las ventanas de la cuestión anterior.


In [67]:
inputs = keras.layers.Input(shape=(5, 1))

gru_1=keras.layers.GRU(8, return_sequences=True, name='gru_1')(inputs)
gru_2=keras.layers.GRU(16, return_sequences=False, name='gru_2')(gru_1)

outputs = layers.Dense(1)(gru_2)

model = keras.Model(inputs=inputs, outputs=outputs)
model.compile(optimizer=keras.optimizers.Adam(), loss='mse')
model.summary()

Model: "functional_11"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_11 (InputLayer)     │ (None, 5, 1)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 5, 8)           │           264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 16)             │         1,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,529 (5.97 KB)

 Trainable params: 1,529 (5.97 KB)

 Non-trainable params: 0 (0.00 B)

In [68]:
es_callback = keras.callbacks.EarlyStopping(
    monitor="val_loss", min_delta=0, patience=10)

history = model.fit(
    X_train, y_train,
    epochs=200,
    validation_split=0.2, shuffle=True, batch_size = 64, callbacks=[es_callback]
)

Epoch 1/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 130.0062 - val_loss: 101.0203
Epoch 2/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 87.6736 - val_loss: 63.0528
Epoch 3/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 55.6523 - val_loss: 44.4481
Epoch 4/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 38.3980 - val_loss: 35.4520
Epoch 5/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 32.2902 - val_loss: 30.3515
Epoch 6/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 28.1143 - val_loss: 26.7344
Epoch 7/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 24.2264 - val_loss: 24.0473
Epoch 8/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 22.7381 - val_loss: 22.0274
Epoch 9/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 21.2251 - val_loss: 20.5502
Epoch 10/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 19.2175 - val_loss: 19.4959
Epoch 11/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 18.2943 - val_loss: 18.6806
Epoch 12/200
38/38 ━━━━━━━━━━━━━━━━━━━━

In [69]:
results = model.evaluate(X_test, y_test, verbose=1)
print('Test Loss: {}'.format(results))

21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 762us/step - loss: 6.7439
Test Loss: 6.967456340789795


<a name='3.3'></a>
## Cuestión 3: Añada más features a la series temporal, por ejemplo `portion_year`. Cree un modelo que mejore al anterior.


In [70]:
## Puede añadir más features
df['portion_year'] = df['Date'].dt.dayofyear / 365.0
df_multi = df[['Temp', 'portion_year']].copy()

## train - test split
train_data = df_multi.iloc[:3000].copy()
test_data = df_multi.loc[3000:, :].copy()
print(train_data.head())
print(test_data.head())

   Temp  portion_year
0  20.7      0.002740
1  17.9      0.005479
2  18.8      0.008219
3  14.6      0.010959
4  15.8      0.013699
      Temp  portion_year
3000  16.9      0.221918
3001  16.5      0.224658
3002  13.6      0.227397
3003  13.2      0.230137
3004   9.4      0.232877


In [71]:
## Create windows

def create_windows_multivariate_np(data, window_size, horizon, target_col_idx, shuffle=False):
    """
    Creates a dataset from the given time series data using NumPy.

    Parameters:
    data (np.ndarray or pd.DataFrame): Time series data with multiple features.
    window_size (int): The number of past time steps to use as input features.
    horizon (int): The number of future time steps to predict.
    target_col_idx (int): The index of the target column in the input data.
    shuffle (bool): Whether to shuffle the data or not.

    Returns:
    tuple: A tuple containing the input-output pairs (X, y) as NumPy arrays.
    """
    if isinstance(data, pd.DataFrame):
        data = data.values

    X, y = [], []
    for i in range(len(data) - window_size - horizon + 1):
        X.append(data[i:i+window_size, :])
        y.append(data[i+window_size+horizon-1, target_col_idx])

    X, y = np.array(X), np.array(y)

    if shuffle:
        indices = np.arange(X.shape[0])
        np.random.shuffle(indices)
        X, y = X[indices], y[indices]

    return X, y

X_train, y_train = create_windows_multivariate_np(train_data,
                                     window_size=past,
                                     horizon=future,
                                     target_col_idx=0,
                                     shuffle=True)
X_test, y_test = create_windows_multivariate_np(test_data,
                                   window_size=past,
                                   horizon=future,
                                   target_col_idx=0,
                                   shuffle=False)

In [72]:
inputs = keras.layers.Input(shape=(5, 2))
lstm_1=keras.layers.LSTM(64, return_sequences=True, name='lstm_1')(inputs)
lstm_2=keras.layers.LSTM(64, return_sequences=True, name='lstm_2')(lstm_1)
lstm_3=keras.layers.LSTM(64, return_sequences=False, name='lstm_3')(lstm_2)
outputs = layers.Dense(1)(lstm_3)
model = keras.Model(inputs=inputs, outputs=outputs)
model.compile(optimizer=keras.optimizers.Adam(), loss='mse')
model.summary()

Model: "functional_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_12 (InputLayer)     │ (None, 5, 2)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 5, 64)          │        17,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 5, 64)          │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 83,265 (325.25 KB)

 Trainable params: 83,265 (325.25 KB)

 Non-trainable params: 0 (0.00 B)

In [73]:
es_callback = keras.callbacks.EarlyStopping(
    monitor="val_loss", min_delta=0, patience=10)

history = model.fit(
    X_train, y_train,
    epochs=200,
    validation_split=0.2, shuffle=True, batch_size = 64, callbacks=[es_callback]
)

Epoch 1/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 100.9826 - val_loss: 16.6321
Epoch 2/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 17.6085 - val_loss: 16.7160
Epoch 3/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 16.5352 - val_loss: 16.6294
Epoch 4/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 16.2535 - val_loss: 16.6076
Epoch 5/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 16.5187 - val_loss: 16.4919
Epoch 6/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 17.0702 - val_loss: 16.1168
Epoch 7/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 16.5818 - val_loss: 11.8444
Epoch 8/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 11.7821 - val_loss: 9.6968
Epoch 9/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 9.6951 - val_loss: 9.2251
Epoch 10/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 10.0451 - val_loss: 9.0877
Epoch 11/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 9.2144 - val_loss: 8.8994
Epoch 12/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms

In [74]:
results = model.evaluate(X_test, y_test, verbose=1)
print('Test Loss: {}'.format(results))

21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 6.0539
Test Loss: 6.366916179656982


<a name='3.4'></a>
## Cuestión 4: ¿En cuáles de estas aplicaciones se usaría un arquitectura 'many-to-one'?

**a)** Clasificación de sentimiento en textos

**b)** Verificación de voz para iniciar el ordenador.

**c)** Generación de música.

**d)** Un clasificador que clasifique piezas de música según su autor.


Respuesta: Se aplica esta arquitecturas en estas aplicaciones:  

**a)** Clasificación de sentimiento en textos  
**b)** Verificación de voz para iniciar el ordenador.  
**d)** Un clasificador que clasifique piezas de música según su autor.  


<a name='3.5'></a>
## Cuestión 5: ¿Qué ventajas aporta el uso de word embeddings?

**a)** Permiten reducir la dimensión de entrada respecto al one-hot encoding.

**b)** Permiten descubrir la similaridad entre palabras de manera más intuitiva que con one-hot encoding.

**c)** Son una manera de realizar transfer learning en nlp.

**d)** Permiten visualizar las relaciones entre palabras con métodos de reducción de dimensioones como el PCA.


Respuesta: Todas son correctas  

**a)** Reducción de dimensión: Sí, tienen muchas menos dimensiones que el one-hot encoding.  
**b)** Similaridad semántica: Sí, capturan la similitud entre palabras (vectores cercanos para palabras similares)  
**c)** Transfer learning: Sí, usar embeddings pre-entrenados transfiere conocimiento de grandes corpus.  
**d)** Visualización: Sí, se pueden usar PCA/t-SNE para visualizar relaciones entre palabras.